# E4: modelo híbrido (apilamiento lineal + CNN de corrección)

1. Regresión ridge por celda y variable con rasgos [1, GFS interpolado en t y t-1 (8), seno/coseno de la hora], ajustada solo con el periodo de entrenamiento.
2. CNN pequeña en 3 km que aprende la corrección sobre la predicción lineal. Entradas: predicción lineal, GFS interpolado t-2..t, DEM, DEM - DEM suavizado 9x9, hora. Pérdida: promedio_v(MSE_v / REF_v^2).
3. Control de sobreajuste: weight decay, pocas épocas, selección de época con el último 15 % del periodo de entrenamiento (no con validación). Dos cortes temporales.
4. Score local y RMSE por variable: lineal sola vs híbrido.

In [1]:
import time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
torch.set_num_threads(2); torch.manual_seed(0); np.random.seed(0)
T0 = time.time()
ROOT = Path('..'); DATA = ROOT/'data'; RES = Path('results'); RES.mkdir(exist_ok=True)
REF = np.array([25.360, 2.0652, 2.6776, 2.0876], dtype=np.float32)
VARS = ['RH','T','U','V']
tr = np.load(DATA/'train.npz'); te = np.load(DATA/'test.npz'); st = np.load(DATA/'static.npz')
X = tr['X'].astype(np.float32); Y = tr['Y'].astype(np.float32); times = tr['times'].astype(str)
Xte = te['X'].astype(np.float32); tte = te['times'].astype(str)
print(X.shape, Y.shape, Xte.shape)

(2935, 17, 13, 4) (2935, 70, 35, 4) (792, 17, 13, 4)


In [2]:
# Interpolación bilineal GFS -> WRF por coordenadas (GFS fila 0 = norte; WRF fila 0 = sur)
glat, glon = st['gfs_lats'], st['gfs_lons']; wlat, wlon = st['wrf_lats'], st['wrf_lons']
def frac_idx(g, w):
    asc = g[1] > g[0]
    gi = g if asc else g[::-1]
    f = np.interp(w, gi, np.arange(len(g)))
    return f if asc else (len(g)-1) - f
fi = frac_idx(glat, wlat); fj = frac_idx(glon, wlon)
W = np.zeros((70*35, 17*13), np.float32)
for a in range(70):
    i0 = int(np.floor(fi[a])); i1 = min(i0+1, 16); di = fi[a]-i0
    for b in range(35):
        j0 = int(np.floor(fj[b])); j1 = min(j0+1, 12); dj = fj[b]-j0
        k = a*35+b
        for ii, wi in ((i0,1-di),(i1,di)):
            for jj, wj in ((j0,1-dj),(j1,dj)):
                W[k, ii*13+jj] += wi*wj
def interp(Xg):  # (N,17,13,4) -> (N,70,35,4)
    return np.einsum('kg,ngv->nkv', W, Xg.reshape(len(Xg), 221, 4)).reshape(len(Xg), 70, 35, 4)
Gi = interp(X); Gte = interp(Xte)

def prev_idx(ts):
    dt = pd.to_datetime(ts, format='%Y%m%d%H')
    p = np.arange(len(ts)) - 1; p[0] = 0
    ok = (dt[1:] - dt[:-1]) == pd.Timedelta(hours=1)
    p[1:][~np.asarray(ok)] = np.arange(1, len(ts))[~np.asarray(ok)]  # si falta t-1, repite t
    return p
p1 = prev_idx(times); p2 = p1[p1]
q1 = prev_idx(tte); q2 = q1[q1]
def hour_feats(ts):
    h = np.array([int(s[-2:]) for s in ts], np.float32)
    return np.stack([np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)], 1)
H = hour_feats(times); Hte = hour_feats(tte)
dem = st['dem'].astype(np.float32)
dem_s = F.avg_pool2d(torch.tensor(dem)[None,None], 9, 1, 4, count_include_pad=False)[0,0].numpy()
print('interp ok', Gi.shape, 'huecos t-1:', (p1 == np.arange(len(times))).sum())

interp ok (2935, 70, 35, 4) huecos t-1: 14


In [3]:
# Ridge por celda (vectorizado): rasgos [1, G_t(4), G_t-1(4), sin, cos]
def lin_feats(G, pa, Hh):
    N = len(G)
    return np.concatenate([np.ones((N,70,35,1),np.float32), G, G[pa],
                           np.broadcast_to(Hh[:,None,None,:], (N,70,35,2))], -1)  # (N,70,35,11)
def fit_ridge(Fe, Yt, lam=1.0):
    mu = Fe.reshape(-1,11).mean(0); sd = Fe.reshape(-1,11).std(0); mu[0]=0; sd[0]=1
    Z = ((Fe-mu)/sd).reshape(len(Fe), -1, 11).astype(np.float64)
    Yc = Yt.reshape(len(Yt), -1, 4).astype(np.float64)
    A = np.einsum('nck,ncl->ckl', Z, Z) + lam*np.eye(11)[None]
    A[:,0,0] -= lam
    B = np.einsum('nck,ncv->ckv', Z, Yc)
    return mu, sd, np.linalg.solve(A, B).astype(np.float32)  # (cells,11,4)
def pred_ridge(m, Fe):
    mu, sd, C = m
    Z = ((Fe-mu)/sd).reshape(len(Fe), -1, 11)
    return np.einsum('nck,ckv->ncv', Z, C).reshape(len(Fe), 70, 35, 4)
def scores(P, Yt):
    rmse = np.sqrt(((P-Yt)**2).mean((0,1,2)))
    return rmse, 1 - (rmse/REF).mean()

In [4]:
# CNN de corrección
class Corr(nn.Module):
    def __init__(s, cin=20, w=32):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(cin,w,3,padding=1), nn.GELU(),
                              nn.Conv2d(w,w,3,padding=1), nn.GELU(),
                              nn.Conv2d(w,w,3,padding=2,dilation=2), nn.GELU(),
                              nn.Conv2d(w,w,3,padding=1), nn.GELU(),
                              nn.Conv2d(w,4,1))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)
    def forward(s, x): return s.net(x)

def build_inputs(Plin, G, pa, pb, Hh, norm):
    gm, gs, pm, ps = norm
    N = len(G)
    parts = [(Plin-pm)/ps, (G[pb]-gm)/gs, (G[pa]-gm)/gs, (G-gm)/gs]
    x = np.concatenate(parts, -1).transpose(0,3,1,2)  # (N,16,70,35)
    d1 = np.broadcast_to(((dem-dem.mean())/dem.std())[None,None], (N,1,70,35))
    d2 = np.broadcast_to(((dem-dem_s)/(dem-dem_s).std())[None,None], (N,1,70,35))
    hh = np.broadcast_to(Hh[:,:,None,None], (N,2,70,35))
    return np.concatenate([x, d1, d2, hh], 1).astype(np.float32)

def train_cnn(Xin, R, Xsel, Rsel, epochs=8, lr=2e-3, wd=1e-2, bs=32, tag=''):
    m = Corr(); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    Xt, Rt = torch.tensor(Xin), torch.tensor(R); Xs, Rs = torch.tensor(Xsel), torch.tensor(Rsel)
    best = (np.inf, None, -1); hist = []
    with torch.no_grad(): l0 = F.mse_loss(torch.zeros_like(Rs), Rs).item()
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xt))
        for k in range(0, len(Xt), bs):
            b = perm[k:k+bs]
            loss = F.mse_loss(m(Xt[b]), Rt[b])  # = mean_v MSE_v/REF_v^2 (residuo dividido por REF)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step(); m.eval()
        with torch.no_grad():
            ls = sum(F.mse_loss(m(Xs[k:k+64]), Rs[k:k+64], reduction='sum').item() for k in range(0,len(Xs),64)) / Rs.numel()
        hist.append(ls)
        if ls < best[0]: best = (ls, {k: v.clone() for k, v in m.state_dict().items()}, ep)
        print(f'{tag} ep{ep} sel_loss={ls:.4f} (sin corr {l0:.4f}) t={time.time()-T0:.0f}s', flush=True)
    m.load_state_dict(best[1]); return m, best[2], l0, hist

def predict_cnn(m, Xin):
    with torch.no_grad():
        return np.concatenate([m(torch.tensor(Xin[k:k+64])).numpy() for k in range(0,len(Xin),64)]).transpose(0,2,3,1)*REF

In [5]:
def run_cut(t_tr_end, t_va_start, t_va_end, tag, epochs=6):
    itr = np.where(times < t_tr_end)[0]; iva = np.where((times >= t_va_start) & (times < t_va_end))[0]
    Fe = lin_feats(Gi, p1, H)
    lm = fit_ridge(Fe[itr], Y[itr])
    Plin = pred_ridge(lm, Fe)  # in-sample en train, fuera de muestra en val
    gm = Gi[itr].mean((0,1,2)); gs = Gi[itr].std((0,1,2))
    norm = (gm, gs, gm, gs)
    Xin = build_inputs(Plin, Gi, p1, p2, H, norm)
    R = ((Y - Plin)/REF).transpose(0,3,1,2).astype(np.float32)
    n_in = int(len(itr)*0.85); ifit, isel = itr[:n_in], itr[n_in:]
    m, bep, l0, hist = train_cnn(Xin[ifit], R[ifit], Xin[isel], R[isel], epochs=epochs, tag=tag)
    Ph = Plin[iva] + predict_cnn(m, Xin[iva])
    r_lin, s_lin = scores(Plin[iva], Y[iva]); r_h, s_h = scores(Ph, Y[iva])
    rows = []
    for name, r, s in (('lineal', r_lin, s_lin), ('hibrido', r_h, s_h)):
        rows.append(dict(corte=tag, modelo=name, train_hasta=t_tr_end, val_desde=t_va_start, val_hasta=t_va_end,
                         n_train=len(itr), n_val=len(iva), epoca_sel=(bep if name=='hibrido' else ''),
                         score=round(float(s),4), **{f'RMSE_{v}': round(float(x),4) for v, x in zip(VARS, r)}))
    print(pd.DataFrame(rows).to_string())
    return rows, bep

In [6]:
# Corte A (oficial local): train < 16 jun, val >= 16 jun
rowsA, bepA = run_cut('2024061600', '2024061600', '2099', 'A_val16jun', epochs=6)

A_val16jun ep0 sel_loss=0.3724 (sin corr 0.3711) t=634s


A_val16jun ep1 sel_loss=0.3693 (sin corr 0.3711) t=705s


A_val16jun ep2 sel_loss=0.3750 (sin corr 0.3711) t=760s


A_val16jun ep3 sel_loss=0.3776 (sin corr 0.3711) t=813s


A_val16jun ep4 sel_loss=0.3781 (sin corr 0.3711) t=872s


A_val16jun ep5 sel_loss=0.3802 (sin corr 0.3711) t=1315s


        corte   modelo train_hasta   val_desde val_hasta  n_train  n_val epoca_sel   score  RMSE_RH  RMSE_T  RMSE_U  RMSE_V
0  A_val16jun   lineal  2024061600  2024061600      2099     2599    336            0.3440  11.1549  1.2400  1.9292  1.8021
1  A_val16jun  hibrido  2024061600  2024061600      2099     2599    336         1  0.3447  11.2235  1.2479  1.9188  1.7903


In [7]:
# Corte B (adicional): train < 26 may, val 26 may - 16 jun
rowsB, bepB = run_cut('2024052600', '2024052600', '2024061600', 'B_val26may', epochs=6)
res = pd.DataFrame(rowsA + rowsB); res.to_csv(RES/'exp04_resultados.csv', index=False); res

B_val26may ep0 sel_loss=0.3676 (sin corr 0.3636) t=1411s


B_val26may ep1 sel_loss=0.3654 (sin corr 0.3636) t=1453s


B_val26may ep2 sel_loss=0.3621 (sin corr 0.3636) t=1493s


B_val26may ep3 sel_loss=0.3571 (sin corr 0.3636) t=1539s


B_val26may ep4 sel_loss=0.3587 (sin corr 0.3636) t=1588s


B_val26may ep5 sel_loss=0.3557 (sin corr 0.3636) t=1635s


        corte   modelo train_hasta   val_desde   val_hasta  n_train  n_val epoca_sel   score  RMSE_RH  RMSE_T  RMSE_U  RMSE_V
0  B_val26may   lineal  2024052600  2024052600  2024061600     2119    480            0.3558  11.7896  1.2266  1.8791  1.7040
1  B_val26may  hibrido  2024052600  2024052600  2024061600     2119    480         5  0.3640  11.7811  1.1729  1.8682  1.6988


,corte,modelo,train_hasta,val_desde,val_hasta,n_train,n_val,epoca_sel,score,RMSE_RH,RMSE_T,RMSE_U,RMSE_V
0,A_val16jun,lineal,2024061600,2024061600,2099,2599,336,,0.3440,11.1549,1.2400,1.9292,1.8021
1,A_val16jun,hibrido,2024061600,2024061600,2099,2599,336,1,0.3447,11.2235,1.2479,1.9188,1.7903
2,B_val26may,lineal,2024052600,2024052600,2024061600,2119,480,,0.3558,11.7896,1.2266,1.8791,1.7040
3,B_val26may,hibrido,2024052600,2024052600,2024061600,2119,480,5,0.3640,11.7811,1.1729,1.8682,1.6988


In [8]:
# Submission solo si el híbrido mejora claramente en ambos cortes (> 0.01 de score)
gain = [res[(res.corte==c)&(res.modelo=='hibrido')].score.item() - res[(res.corte==c)&(res.modelo=='lineal')].score.item() for c in res.corte.unique()]
print('ganancias', gain)
if min(gain) > 0.01:
    Fe = lin_feats(Gi, p1, H); lm = fit_ridge(Fe, Y); Plin = pred_ridge(lm, Fe)
    gm = Gi.mean((0,1,2)); gs = Gi.std((0,1,2)); norm = (gm, gs, gm, gs)
    Xin = build_inputs(Plin, Gi, p1, p2, H, norm); R = ((Y - Plin)/REF).transpose(0,3,1,2).astype(np.float32)
    ep_final = max(bepA, bepB) + 1
    n_in = int(len(Y)*0.9)
    m, _, _, _ = train_cnn(Xin[:n_in], R[:n_in], Xin[n_in:], R[n_in:], epochs=ep_final, tag='final')
    Fte = lin_feats(Gte, q1, Hte); Plt = pred_ridge(lm, Fte)
    Pte = Plt + predict_cnn(m, build_inputs(Plt, Gte, q1, q2, Hte, norm))
    sub = pd.read_csv(DATA/'sample_submission.csv', usecols=['id'])
    tid = {t: i for i, t in enumerate(tte)}
    parts = sub['id'].str.split('_', expand=True)
    ti = parts[0].map(tid).values; ri = parts[1].astype(int).values; ci = parts[2].astype(int).values
    vals = Pte[ti, ri, ci]
    vals[:,0] = np.clip(vals[:,0], 0, 100)
    out = pd.DataFrame({'id': sub['id'], **{v: np.round(vals[:,k], 3) for k, v in enumerate(VARS)}})
    out.to_csv(RES/'exp04_submission.csv', index=False); print('submission', out.shape)
else:
    print('Sin mejora clara: no se genera submission')
print(f'tiempo total {time.time()-T0:.0f}s')

ganancias [0.0007000000000000339, 0.008199999999999985]
Sin mejora clara: no se genera submission
tiempo total 1639s
